# Three-Phase Power

In [1]:
from typing import Literal, cast

import matplotlib.pyplot as plt
import numpy as np
import schemdraw
import schemdraw.elements as elm
from matplotlib.axes import Axes

from src.drawing_utils import Arrow, Point, clear_axes
from src.figure_utils import LETTERS, process_figure
from src.plotting_utils import (
    configure_matplotlib,
    format_xaxis,
    rm,
    set_discrete_colors,
)
from src.schemdraw_utils import (
    Cardinal,
    ParallelLines,
    SinglePhaseLines,
    ThreePhaseLines,
    configure_schemdraw,
)

configure_matplotlib()
configure_schemdraw()

## Introduction to three-phase power

The means of generation, transfer, and consumption of AC power that we have discussed so far is known as *single-phase* power. If not already familiar with three-phase power, you will be surprised to learn that single-phase power is not the main way that power is transferred in the electrical grid. In a single-phase system, a sinusoidal voltage $v(t)$ is generated between a set of two power lines such that power is transferred to loads placed across those lines (in parallel). This is illustrated by Figure 5.1(a). As we learned in the previous chapter, the power delivered by a single-phase circuit oscillates sinusoidally with time, which can be impractical.

On the other hand, three-phase power, sometimes abbreviated $3 \phi$, is a way of transmitting power for which the power delivered is constant over time. Steady power is advantageous for heavy industrial loads, such as large electric motors, and for long-distance transmission. This is one reason why three-phase power is carried by the grid's transmission lines, besides the fact that three-phase power can reduce the amount of conductor material needed and thus results in reduced infrastructure costs.

A three-phase circuit has three voltage lines, each of which carries a sinusoidally oscillating voltage. The three voltages are called *phase voltages* and are denoted $v_a(t)$, $v_b(t)$, and $v_c(t)$. The three voltage sinusoids are of equal magnitude $|V|$, but their phase angles are one third of a cycle ($2 \pi / 3$ radians) apart, meaning that the three voltage sinusoids sum to zero at any given time. The three phase voltages can be represented by phasors as in Figure 5.2.

The generators that supply the phase voltages (and thus power the circuit) are called three-phase generators, and the loads that are powered by the circuit are called three-phase loads. Each of these generators and loads is connected to all three lines. This is illustrated by Figure 5.1(b). Each load actually consists of three impedances. If a given load's three impedances are of equal value ($Z$), then the load is called a *balanced* three-phase load. A three-phase circuit in which all loads are balanced is called a balanced three-phase circuit.

For a balanced load, the fact that the voltage sinusoids sum to zero also means that the sinusoidal currents through the three impedances sum to zero. In other words, the three currents cancel out. Thus, no extra line is required to carry current back to the voltage source, as is required in a single-phase circuit. This is why a balanced three-phase circuit can transfer the same amount of power as a single-phase circuit, but with less conductor material.

In [ ]:
def generate_phase_angles(n: int) -> list[float]:
    return cast(list[float], np.linspace(0, 2 * np.pi, n + 1).tolist())[:-1]


def draw(
    ax1: Axes,
    ax2: Axes,
    n_phases: Literal[1, 3],
    element_scale: float = 1.0,
) -> None:
    title = {
        1: "(a) Single-phase",
        3: "(b) Three-phase",
    }[n_phases]
    ax1.set_title(rm(title))
    clear_axes(ax1)
    with schemdraw.Drawing(canvas=ax1) as d:
        w = 1.5
        length = 6
        horizontal_length = 6
        generator_label = {
            1: "$v(t)$",
            3: "$v_a(t)$ \n $v_b(t)$ \n $v_c(t)$",
        }[n_phases]
        load_label = {
            1: "$Z$",
            3: "$Z$ \n $Z$ \n $Z$",
        }[n_phases]
        parallel_lines_class: type[ParallelLines] = {
            1: SinglePhaseLines,
            3: ThreePhaseLines,
        }[n_phases]
        if issubclass(parallel_lines_class, ThreePhaseLines):
            horizontal_length -= w
        pl = parallel_lines_class(d, w, element_scale=element_scale)
        pl.draw_element(elm.SourceSin, Cardinal.LEFT, generator_label)
        pl.draw_lines(Cardinal.RIGHT, horizontal_length)
        pl.draw_node()
        d.push()
        pl.draw_lines(Cardinal.UP, length / 2)
        pl.draw_lines(Cardinal.RIGHT, horizontal_length)
        pl.draw_element(elm.Resistor, Cardinal.RIGHT, load_label)
        d.pop()
        pl.draw_lines(Cardinal.DOWN, length / 2)
        pl.draw_node()
        d.push()
        pl.draw_lines(Cardinal.RIGHT, horizontal_length)
        pl.draw_element(elm.Resistor, Cardinal.RIGHT, load_label)
        d.pop()
        pl.draw_lines(Cardinal.DOWN, length / 2)
        pl.draw_node()
        d.push()
        pl.draw_lines(Cardinal.DOWN, length / 2)
        pl.draw_lines(Cardinal.RIGHT, horizontal_length)
        pl.draw_element(elm.Resistor, Cardinal.RIGHT, load_label)
        d.pop()
        pl.draw_lines(Cardinal.LEFT, horizontal_length)
        pl.draw_element(elm.SourceSin, Cardinal.LEFT, generator_label)

    xs = np.linspace(-np.pi / 2, 3 * np.pi / 2, 500)
    for i, theta in enumerate(generate_phase_angles(n_phases)):
        label = {
            1: "$v(t)$",
            3: f"$v_{LETTERS[i]}(t)$",
        }[n_phases]
        ax2.plot(xs, np.cos(xs + theta), label=label)
    set_discrete_colors(ax2)
    format_xaxis(ax2, x_texts=True)
    ax2.set_xlabel(r"$\omega t$")
    ax2.set_yticks([0])
    ax2.grid(linestyle="--", axis="x")
    ax2.legend(loc="upper right")


fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(6.4, 5.4),
    layout="tight",
    gridspec_kw=dict(height_ratios=[1.5, 1]),
)

draw(ax1, ax3, n_phases=1)
draw(ax2, ax4, n_phases=3, element_scale=0.6)

process_figure(fig, 5, 1, label_subfigures="no")

```{figure} img/fig_5_1.png
:align: center
:width: 64%
```

<p></p>

In [ ]:
def draw_n_phasor_diagram(
    ax: Axes,
    angles: list[float],
    real_axis_lim: tuple[float, float] = (-5, 6),
    imag_axis_lim: tuple[float, float] = (-5, 6.5),
) -> None:
    clear_axes(ax)

    ax.set_xlim(real_axis_lim)
    ax.set_ylim(imag_axis_lim[0], imag_axis_lim[1] + 1)

    real_axis = Arrow(Point(real_axis_lim[0], 0), Point(real_axis_lim[1], 0)).drawn(ax)
    real_axis.end.labeled(ax, rm("Re"), (7.5, -3))
    imag_axis = Arrow(Point(0, imag_axis_lim[0]), Point(0, imag_axis_lim[1])).drawn(ax)
    imag_axis.end.labeled(ax, rm("Im"), (0, 2.5))

    origin = Point(0, 0)
    origin.drawn(ax)
    for i, angle in enumerate(angles):
        arrow = Arrow.from_polar(origin, length=4, angle=angle).drawn(ax, linewidth=1.5)
        phase_letter = LETTERS[i]
        arrow.rotated(0.3).end.labeled(ax, f"$V_{phase_letter}$")


fig, ax = plt.subplots(figsize=(3.6, 3.6), layout="tight")

draw_n_phasor_diagram(
    ax,
    angles=generate_phase_angles(3),
    real_axis_lim=(-5.5, 5.5),
    imag_axis_lim=(-5, 5),
)

process_figure(fig, 5, 2, label_subfigures="no")

```{figure} img/fig_5_2.png
:align: center
:width: 36%
```

<p></p>

## Comparison of *n*-phase circuits

Circuits with a number of phases other than one or three are possible. Circuits with more than one phase are known as polyphase systems. But why is three-phase power more common than other polyphase systems, and more common than even single-phase systems? Why not two-phase power, or four? This section will compare circuits with an arbitrary number of phases $n$. We will determine the advantages and disadvantages of circuits with different numbers of phases. We will assume balanced three-phase loads with, for simplicity, impedance value $R$. However, the conclusions would be the same for an arbitrary impedance $Z$.

Consider a **circuit with a single phase** $a$, such as that shown in Figure 5.3(a). It has one impedance $R_a = R$, supplied by one phase voltage:

$$
v_a(t) = |V| \cos \omega t
$$

Recall the instantaneous power delivered over time in a single-phase circuit from [Chapter 4](4_complex_power_and_the_power_factor.ipynb). In a single-phase circuit, the total power $p_{\, \text{tot.}}(t)$ delivered in the circuit is just the power $p_a(t)$ delivered to impedance $R_a$, the time-average of which is $\bar{p}_{\, \text{tot.}} = |V|^2 / \, 2 R$.

$$
p_{\, \text{tot.}}(t)
= p_a(t)
= \frac{|V|^2}{R}
= \frac{|V|^2}{R} \cos^2 \omega t
= \underbrace{\, \frac{1}{2} \frac{|V|^2}{R}}_{\bar{p}_{\, \text{tot.}}} + \frac{1}{2} \frac{|V|^2}{R} \cos 2 \omega t
$$ (single_phase_p)

Similarly, the sum of the phase currents $i_{\, \text{tot.}}(t)$ is just the current $i_a(t)$ delivered through impedance $R_a$, as follows:

$$
i_{\, \text{tot.}}(t)
= i_a(t)
= \frac{|V|}{R} \cos \omega t
$$

Of course, the sum of the phase currents is nonzero (except where it crosses the $\omega t$-axis) and we need a 'return' line (the neutral line) connected to the same 0-V potential as voltage source $v_a$. While these conclusions may seem obvious for a single-phase circuit, we will see that they do not necessarily apply to higher-phase circuits.

In a **two-phase circuit** such as in Figure 5.3(b), we have two impedances, $R_a = R_b = R$. The impedances' respective voltages are $v_a(t)$ and $v_b(t)$, whose magnitudes are both $|V|$ and whose phases are $\pi / 2$ radians out-of-phase relative to each other:

$$
\begin{aligned}
& v_a(t) = |V| \cos \omega t \\
& v_b(t) = |V| \cos (\omega t + \pi / 2)
\end{aligned}
$$

``` {admonition} In a two-phase circuit, why are the phase voltages out-of-phase by $\pi / 2$ radians?
Looking at a three-phase circuit and trying to simplify it to two phases, one might expect the two phase voltages to be out-of-phase by $\pi$ radians such that they sum to zero. However, this is not what we mean when we talk about two-phase circuits. The two phase voltages being out-of-phase by $\pi$ radians would just be a version of the single-phase circuit in which $|V_a|$ is replaced by $|V| / 2$ and the 0-V potential is replaced by $- |V| / 2$.
```

The total power delivered in the two-phase circuit is the sum of power delivered to both loads, as follows. Note that it is no longer sinusoidal, as in was in the single-phase circuit, but *constant*. In fact, the time-average total power delivered by any circuit with $n \geq 2$ phases is always constant, making such circuits strong contenders for being able to deliver a steady supply of total power in a power system.

$$
\begin{aligned}
p_{\, \text{tot.}}(t)
& = p_a(t) + p_b(t) \\
& = \frac{V_a^2 + V_b^2}{R} \\
& = \frac{(|V| \cos \omega t)^2 + \left( |V| \cos \! \left( \omega t + \frac{\pi}{2} \right) \right)^2}{R} \\
& = \frac{|V|^2 ((\cos \omega t)^2 + (- \sin \omega t)^2)}{R} \qquad \text{Apply Pythagorean identity, } \sin^2 x + \cos^2 x = 1. \\
& = \underbrace{\frac{|V|^2}{R}}_{\bar{p}_{\, \text{tot.}}}
\end{aligned}
$$

Just as in the single-phase circuit, the sum of the phase currents in the two-phase circuit is nonzero and a neutral line is indeed needed:

$$
i_{\, \text{tot.}}(t)
= i_a(t) + i_b(t)
= |V| \cos \omega t + |V| \cos \! \left( \omega t + \frac{\pi}{2} \right)
= \sqrt{2} \, \frac{|V|}{R} \cos \! \left( \omega t + \frac{\pi}{4} \right)
$$

In a **three-phase circuit** such as in Figure 5.3(c), the three phase voltages are $2 \pi / 3$ radians out-of-phase relative to each other:

$$
\begin{aligned}
& v_a(t) = |V| \cos \omega t \\
& v_b(t) = |V| \cos (\omega t + 2 \pi / 3) \\
& v_c(t) = |V| \cos (\omega t - 2 \pi / 3)
\end{aligned}
$$

Just as in the two-phase circuit, the total power delivered in the three-phase circuit is constant:

$$
p_{\, \text{tot.}}(t)
= p_a(t) + p_b(t) + p_c(t)
= \underbrace{\, \frac{3}{2} \frac{|V|^2}{R}}_{\bar{p}_{\, \text{tot.}}}
$$

However, unlike the single- and two-phase circuits, the sum of the phase currents in the three-phase circuit is zero at all times (as long as the loads are balanced). Thus, no neutral line is needed. In fact, no neutral line is needed for any circuit with $n \geq 3$ phases. This means that a three-phase circuit can deliver $3$ times as much power as a single-phase circuit using only $1.5$ times as many lines and only $1.5$ times as much conductor material.

Indeed, in **circuits with more than three phases**, such as in Figure 5.3(d) and (e), the time-average total power delivered is constant and no neutral line is needed&mdash;the same as for a three-phase circuit. Circuits with $n > 3$ phases have no advantage over three-phase circuits and are avoided because they need more conductors that would have to be installed and maintained in the power system (even if they need the same number of conductors per unit power). Thus, three-phase power is the clear winner.

The following table summarizes circuits with different numbers of phases, in terms of the minimum number of lines needed, whether the total power is sinusoidal or constant, the time-average total power, and the time-average total power per line.

| $n$            | Min. # of lines | Total power, $p_{\, \text{tot.}}(t)$  | Avg. total power, $\bar{p}_{\, \text{tot.}}$ | Avg. total power per line |
|----------------|-----------------|---------------------------------------|----------------------------------------------|---------------------------|
| 1              | 2               | Sinusoidal (Eq. {eq}`single_phase_p`) | $= \|V\|^2 / \, 2 R$                         | $= \|V\|^2 / \, 4 R$      |
| 2              | 3               | Constant                              | $= \|V\|^2 / R$                              | $= \|V\|^2 / \, 3 R$      |
| 3              | 3               | Constant                              | $= 3 \|V\|^2 / \, 2 R$                       | $= \|V\|^2 / \, 2 R$      |
| 4              | 4               | Constant                              | $= 2 \|V\|^2 / R$                            | $= \|V\|^2 / \, 2 R$      |
| Any $n \geq 3$ | $n$             | Constant                              | $= n \|V\|^2 / \, 2 R$                       | $= \|V\|^2 / \, 2 R$      |

In [ ]:
def draw(ax0: Axes, ax1: Axes, ax2: Axes, ax3: Axes, angles: list[float]) -> None:
    n_phases = len(angles)
    title = f"{n_phases} phases" if n_phases > 1 else "Single phase"
    ax0.set_title(rm(f"({LETTERS[n_phases - 1]}) {title}"), loc="left")

    clear_axes(ax0)

    ax0.set_xlim((-5, 5))
    ax0.set_ylim((-4, 5))

    with schemdraw.Drawing(canvas=ax0):
        origin = (0.0, (-1.5 if n_phases == 1 else 0.0))
        elm.Dot().at(origin)
        if n_phases < 3:
            elm.Ground()
        for i, angle in enumerate(angles):
            elm.Resistor().at(origin).length(4).theta(np.rad2deg(angle)).label(
                r"\small $R$"
            )
            phase_letter = LETTERS[i]
            elm.Dot().label(rf"\small $v_{phase_letter}(t)$")

    draw_n_phasor_diagram(ax1, angles)

    ts = np.linspace(0, 2 * np.pi)

    origin = Point(0, 0)
    origin.drawn(ax1)
    for i, angle in enumerate(angles):
        phase_letter = LETTERS[i]
        label = f"${phase_letter}$"
        ax2.plot(ts, np.cos(ts + angle))
        ax3.plot(ts, np.square(np.cos(ts + angle)), label=label)

    ax2.set_ylim((-1.1, 1.1))
    ax2.set_yticks([-1, 0, 1])

    ax3.plot(
        ts,
        sum(np.square(np.cos(ts + angle)) for angle in angles),
        "k",
        label=rm("total"),
    )
    ax3.set_ylim((-0.1, 3))

    for ax in [ax2, ax3]:
        format_xaxis(ax, x_texts=True, start_at=0)

    ax3.legend(loc="upper right", prop=dict(size=6))

    if angles == [0.0]:
        ax2.set_title("$v_i(t) / |V|$")
        ax3.set_title("$p_i(t) / (|V|^2 / R)$")


fig, axs = plt.subplots(
    nrows=5, ncols=4, figsize=(6.5 / 0.64, 9 / 0.64), layout="constrained"
)
# Single-phase:
draw(*axs[0], angles=[0.0])
# Two-phase:
draw(*axs[1], angles=[0.0, np.pi / 2])
# Three-phase:
draw(*axs[2], angles=generate_phase_angles(3))
# Four-phase:
draw(*axs[3], angles=generate_phase_angles(4))
# Five-phase:
draw(*axs[4], angles=generate_phase_angles(5))
for col in [2, 3]:
    axs[4][col].set_xlabel(r"$\omega t$")

process_figure(fig, 5, 3, label_subfigures="no")

```{figure} img/fig_5_3.png
:align: center
:width: 100%
```

<p></p>

## Wye versus delta configurations

So far, we have assumed three-phase generators and loads to be in the $\mathrm{Y}$ configuration (pronounced, and often written as, "wye"). Also called the "star" configuration, the wye configuration includes a 0-V neutral node which we will denote $n$. Each of the three-phase lines $a$, $b$, and $c$ are connected to $n$. In a three-phase generator following the wye configuration, each of the three lines is connected to node $n$ through a voltage source $v_i$, resulting in the three phase voltages $v_i(t)$. In a wye-configured three-phase load, each of the three lines is connected to node $n$ through an impedance $Z$. If the load is unbalanced, a fourth, return line is required to carry extra current at node $n$ of the load back to node $n$ of the generator. <!-- What if the generator is delta? -->

There is a second three-phase configuration called $\Delta$, or delta. In the delta configuration, components&mdash;whether voltage sources or impedances&mdash;are not connected between each line and neutral. Instead, components are connected across each pair of lines: $(a, b)$, $(b, c)$, and $(c, a)$. There is no node $n$ and no possibility of a return line, so loads following the delta configuration must be balanced. The voltage across each component is the difference between the voltages of the two lines to which it is connected, e.g., $v_{ba}(t) = v_b(t) - v_a(t)$. The magnitude of the voltage across each component is $\sqrt{3} |V|$, or $\sqrt{3}$ times as much that of a wye-connected counterpart.

Both wye- and delta-configured generators and loads can be connected to the same three-phase circuit.

In [ ]:
short = 1
long = 3


def wye(
    ax: Axes,
    elm_class: type[elm.Element2Term],
) -> None:
    clear_axes(ax)
    with schemdraw.Drawing(canvas=ax) as d:
        left_or_right = {elm.SourceSin: "left", elm.Resistor: "right"}[elm_class]
        degrees = {elm.SourceSin: 0, elm.Resistor: 60}[elm_class]
        d.push()
        elm.Dot().label("$a$")
        getattr(elm.Line().length(short), left_or_right)()
        getattr(elm_class().length(long), left_or_right)()
        elm.Dot().label(
            "$n$", ofst=(0.2 * {elm.SourceSin: 1, elm.Resistor: -1}[elm_class], 0)
        )
        d.pop()
        d.push()
        d.move(0, long * np.sqrt(3) / 2)
        elm.Dot().label("$b$", loc="bottom")
        getattr(elm.Line().length(short + long + long / 2), left_or_right)()
        elm_class().length(long).theta(-1 * (60 + degrees))
        d.pop()
        d.move(0, -1 * long * np.sqrt(3) / 2)
        elm.Dot().label("$c$")
        getattr(elm.Line().length(short + long + long / 2), left_or_right)()
        elm_class().length(long).theta(60 + degrees)


def delta(
    ax: Axes,
    elm_class: type[elm.Element2Term],
) -> None:
    clear_axes(ax)
    with schemdraw.Drawing(canvas=ax) as d:
        left_or_right = {elm.SourceSin: "left", elm.Resistor: "right"}[elm_class]
        degrees = {elm.SourceSin: 0, elm.Resistor: 120}[elm_class]
        d.push()
        elm.Dot().label("$a$")
        getattr(elm.Line().length(short), left_or_right)()
        elm.Dot()
        elm_class().length(long * np.sqrt(3)).theta(150 - degrees)
        d.pop()
        d.push()
        d.move(0, long * np.sqrt(3) / 2)
        elm.Dot().label("$b$", loc="bottom")
        getattr(elm.Line().length(short + long + long / 2), left_or_right)()
        elm.Dot()
        elm_class().length(long * np.sqrt(3)).down()
        d.pop()
        d.move(0, -1 * long * np.sqrt(3) / 2)
        elm.Dot().label("$c$")
        getattr(elm.Line().length(short + long + long / 2), left_or_right)()
        elm.Dot()
        elm_class().length(long * np.sqrt(3)).theta(30 + degrees)


fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
    nrows=2, ncols=2, figsize=(6.4, 6.4), layout="tight"
)

elm.style(elm.STYLE_IEC)
wye(ax1, elm_class=elm.SourceSin)
ax1.set_title("Wye-configured sources")
wye(ax2, elm_class=elm.Resistor)
ax2.set_title("Wye-configured impedances")
delta(ax3, elm_class=elm.SourceSin)
ax3.set_title("Delta-configured sources")
delta(ax4, elm_class=elm.Resistor)
ax4.set_title("Delta-configured impedances")
elm.style(elm.STYLE_IEEE)

process_figure(fig, 5, 4, label_subfigures="letter_only")

```{figure} img/fig_5_4.png
:align: center
:width: 64%
```

<p></p>